# 023 — Training: unet_v2 and unet_restormer

Trains `unet_v2` and `unet_restormer` — two independent architectural variants of `unet`, each its own architecture name so neither collides with `unet`'s checkpoint or each other's:

- `unet_v2` (`scripts/unet_v2.py`) — the standard UNet with three independently-toggleable modifications, all off by default and matching `unet.py` exactly when off: `use_strided_conv` (learned stride-2 convolution instead of `MaxPool2D`), `use_upsample_conv` (bilinear upsample + `Conv2D` instead of `Conv2DTranspose`, avoids checkerboard artifacts), `dropout_rate` (`SpatialDropout2D` at the bottleneck and first decoder block only — see `scripts/unet_v2.py`'s module docstring for why).
- `unet_restormer` (`scripts/unet_restormer.py`) — the standard UNet with a Restormer transformer block (Zamir et al. 2022) inserted at the bottleneck, giving the network a global receptive field via channel-wise self-attention (linear cost in image size, not quadratic) — see `code-review.md` §7.3.

This split out of `022_training_v2.ipynb`'s former Part B, which mixed this deterministic-architecture job with an unrelated beta-NLL retraining job in one notebook. Same block layout as `020`/`021`/`022` — see `020_training.ipynb`'s title cell for the full table of companion notebooks. Does not modify or retrain `unet`, `resunet`, `attention_unet`, or `efficientnet_unet`.

Only the **artwork-and-mockups** split is used (see §1).


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.


In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check.


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import compile_model, get_callbacks, get_model
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Same split as `020`/`021`/`022` §1 — required so these checkpoints are trained and evaluated under the same conditions as `unet`'s, which they are compared against.


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss function

Neither `unet_v2` nor `unet_restormer` is in `scripts.trainer._ADVANCED_LOSS_ARCHS` (only `efficientnet_unet` is), so both compile with the same `combined_loss` (MAE + (1 − SSIM)) as `unet` — the direct baseline they are meant to be compared against. See `020_training.ipynb` §2 for the Laplacian/FFT loss used by `efficientnet_unet` only.


## 3. Train both variants

`DET_VARIANTS` fixes each architecture's constructor kwargs (the enabled modifications for `unet_v2`; `num_heads`/`ffn_expansion_factor` for `unet_restormer`, both already its defaults — spelled out here for the record). Checkpoints go to `models/deterministic/<arch>/best_model.keras`, same tree as `020`. Set `EPOCHS = 2` for a quick smoke test before committing to a full run.


In [ ]:
DET_VARIANTS = {
    "unet_v2": dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2),
    "unet_restormer": dict(num_heads=8, ffn_expansion_factor=2),
}
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
MODEL_DIR = settings.MODELS_DIR / "deterministic"
LOG_DIR = settings.LOGS_DIR / "deterministic"

histories: dict = {}

for arch, kwargs in DET_VARIANTS.items():
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}  ({kwargs})")
    print(f"{'=' * 60}")

    model = get_model(arch, **kwargs)
    model = compile_model(
        model, arch, lr=settings.LEARNING_RATE, loss_alpha=settings.LOSS_ALPHA
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(arch, log_dir=LOG_DIR, model_dir=MODEL_DIR)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

## 4. Training curves


In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

## 5. Summary

Checkpoints saved to `models/deterministic/<arch>/best_model.keras`. Logs written to `logs/deterministic/<arch>/`.


In [ ]:
for arch in DET_VARIANTS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")